**Assessment 1: English–Hindi Dataset Processing and Analysis**


In [31]:
!git lfs install
!git clone https://huggingface.co/datasets/ainlpml/english-hindi

Git LFS initialized.
Cloning into 'english-hindi'...
remote: Enumerating objects: 17, done.
remote: Total 17 (delta 0), reused 0 (delta 0), pack-reused 17 (from 1)
Receiving objects: 100% (17/17), 1023.20 KiB | 5.04 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [32]:
from datasets import load_dataset

ds = load_dataset("ainlpml/english-hindi")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 20000
    })
})


In [34]:
!ls -R

.:
 english-hindi		        english_hindi_10000.csv
'english_hindi_10000 (1).csv'   sample_data

./english-hindi:
eng.txt  hin.txt  README.md

./sample_data:
anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md


In [35]:
!ls -R english-hindi

english-hindi:
eng.txt  hin.txt  README.md


In [36]:
import pandas as pd

# Read English sentences
with open("english-hindi/eng.txt", "r", encoding="utf-8") as f:
    english_sentences = f.readlines()

# Read Hindi sentences
with open("english-hindi/hin.txt", "r", encoding="utf-8") as f:
    hindi_sentences = f.readlines()

# Create DataFrame
df = pd.DataFrame({
    "English Sentences": [x.strip() for x in english_sentences],
    "Hindi Sentences": [x.strip() for x in hindi_sentences]
})

# Check data
print(df.head())
print("Total rows:", len(df))

                                   English Sentences  \
0  However, Paes, who was partnering Australia's ...   
1  Whosoever desires the reward of the world, wit...   
2  "The value of insects in the biosphere is enor...   
3  Mithali To Anchor Indian Team Against Australi...   
4  After the assent of the Honble President on 8t...   

                                     Hindi Sentences  
0  आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाल...  
1  और जो शख्स (अपने आमाल का) बदला दुनिया ही में च...  
2  जैव-मंडल में कीड़ों का मूल्य बहुत है, क्योंकि ...  
3    आस्ट्रेलिया के खिलाफ वनडे टीम की कमान मिताली को  
4  8 सितम्‍बर, 2016 को माननीय राष्‍ट्रपति की स्‍व...  
Total rows: 10000


In [37]:
df = df.head(10000)

print("Rows after selection:", len(df))

Rows after selection: 10000


In [38]:
df["Word Count (English)"] = (
    df["English Sentences"]
    .str.split()
    .str.len()
)

df["Word Count (Hindi)"] = (
    df["Hindi Sentences"]
    .str.split()
    .str.len()
)

In [39]:
df = df[
    (df["Word Count (English)"].between(5, 50)) &
    (df["Word Count (Hindi)"].between(5, 50))
]

print("Rows after word-count filtering:", len(df))

Rows after word-count filtering: 8788


In [40]:
df["Difference between Word Count (English) and Word Count (Hindi)"] = (
    df["Word Count (English)"] -
    df["Word Count (Hindi)"]
)

In [41]:
df = df[
    df["Difference between Word Count (English) and Word Count (Hindi)"]
    .between(-10, 10)
]

print("Final rows:", len(df))

Final rows: 8214


In [42]:
!pip install openpyxl

In [43]:
df.to_excel(
    "assignment1_cleaned_dataset.xlsx",
    index=False
)

print("Excel file saved successfully!")

Excel file saved successfully!


**Assessment No. 2 – Translation with LLM**


In [44]:
!pip install transformers sentencepiece sacrebleu openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.9 MB/s eta 0:00:00


In [45]:
import pandas as pd

# Load Assignment 1 output
df = pd.read_excel("assignment1_cleaned_dataset.xlsx")

# Select first 100 sentences
df_100 = df.head(100)

print(df_100.head())

                                   English Sentences  \
0  However, Paes, who was partnering Australia's ...   
1  Whosoever desires the reward of the world, wit...   
2  "The value of insects in the biosphere is enor...   
3  Mithali To Anchor Indian Team Against Australi...   
4  After the assent of the Honble President on 8t...   

                                     Hindi Sentences  Word Count (English)  \
0  आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाल...                    23   
1  और जो शख्स (अपने आमाल का) बदला दुनिया ही में च...                    26   
2  जैव-मंडल में कीड़ों का मूल्य बहुत है, क्योंकि ...                    21   
3    आस्ट्रेलिया के खिलाफ वनडे टीम की कमान मिताली को                     9   
4  8 सितम्‍बर, 2016 को माननीय राष्‍ट्रपति की स्‍व...                    18   

   Word Count (Hindi)  \
0                  28   
1                  35   
2                  22   
3                   9   
4                  19   

   Difference between Word Count (English) 

In [49]:
!pip install transformers sentencepiece sacrebleu openpyxl -q

In [53]:
import pandas as pd

# Load Assignment 1 Excel file
df = pd.read_excel("assignment1_cleaned_dataset.xlsx")

# Take first 100 English-Hindi sentence pairs
df_100 = df.head(100)

print(df_100.head())

                                   English Sentences  \
0  However, Paes, who was partnering Australia's ...   
1  Whosoever desires the reward of the world, wit...   
2  "The value of insects in the biosphere is enor...   
3  Mithali To Anchor Indian Team Against Australi...   
4  After the assent of the Honble President on 8t...   

                                     Hindi Sentences  Word Count (English)  \
0  आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाल...                    23   
1  और जो शख्स (अपने आमाल का) बदला दुनिया ही में च...                    26   
2  जैव-मंडल में कीड़ों का मूल्य बहुत है, क्योंकि ...                    21   
3    आस्ट्रेलिया के खिलाफ वनडे टीम की कमान मिताली को                     9   
4  8 सितम्‍बर, 2016 को माननीय राष्‍ट्रपति की स्‍व...                    18   

   Word Count (Hindi)  \
0                  28   
1                  35   
2                  22   
3                   9   
4                  19   

   Difference between Word Count (English) 

In [54]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-hi"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Model loaded successfully!


In [55]:
translations = []

for sentence in df_100["English Sentences"]:

    # Tokenize input
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Generate translation
    outputs = model.generate(**inputs)

    # Decode output
    translated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    translations.append(translated_text)

print("Translation completed!")

Translation completed!


In [56]:
df_100["Model-generated Hindi Translation"] = translations

/tmp/ipykernel_1370/3441559154.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_100["Model-generated Hindi Translation"] = translations


In [57]:
output_df = df_100[
    ["English Sentences", "Model-generated Hindi Translation"]
]

output_df.columns = [
    "Original English Sentence",
    "Model-generated Hindi Translation"
]

output_df.to_excel(
    "assignment2_translation.xlsx",
    index=False
)

print("assignment2_translation.xlsx saved successfully!")

assignment2_translation.xlsx saved successfully!


In [58]:
import sacrebleu

# Reference translations (ground truth)
references = df_100["Hindi Sentences"].tolist()

# Model predictions
predictions = df_100["Model-generated Hindi Translation"].tolist()

# BLEU Score
bleu = sacrebleu.corpus_bleu(predictions, [references])

# CHRF Score
chrf = sacrebleu.corpus_chrf(predictions, [references])

# TER Score
ter = sacrebleu.corpus_ter(predictions, [references])

print("BLEU Score :", bleu.score)
print("CHRF Score :", chrf.score)
print("TER Score  :", ter.score)

BLEU Score : 15.506551298490983
CHRF Score : 36.29121430752617
TER Score  : 83.5171568627451


In [59]:
with open("evaluation_scores.txt", "w", encoding="utf-8") as f:
    f.write(f"BLEU Score: {bleu.score:.2f}\n")
    f.write(f"CHRF Score: {chrf.score:.2f}\n")
    f.write(f"TER Score: {ter.score:.2f}\n")

print("evaluation_scores.txt saved successfully!")

evaluation_scores.txt saved successfully!


In [60]:
from google.colab import files

files.download("assignment2_translation.xlsx")
files.download("evaluation_scores.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>